[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/bezier/bezier.ipynb)

# The Cone of a Rational Bézier Curve

This example computes the cone swept by the rays of a rational quadratic Bézier curve and its projection onto the unit sphere. A rational Bézier curve sums its control points, each scaled by its weight and by a Bernstein polynomial of the curve parameter, and divides by the summed weights. Swept through its middle weight, a quadratic curve passes from an ellipse through a parabola into a hyperbola, and negating both end weights traces the complementary arc of the same conic.

With the plane modelled at unit height in three-dimensional space, each weighted point sits along its ray from the origin as a three-dimensional antivector, so the rational curve reduces to a plain polynomial Bézier sum of antivectors without division. Each antivector represents its ray through the origin, which pierces the unit sphere at two antipodal points with the line at infinity along the equator. Joining two control points with `p & q` forms the plane through the origin and their connecting line, and the three sides of the control triangle assemble a symmetric duality map `Conic` (`Plane+ <- Point-`) whose zero set `conic(ray) & ray == 0` is the cone of the conic. Representing the rational curve directly as a sum of weighted homogeneous points follows [Harquin, Breuils, Biri and Nozick, *Projective Geometric Algebra powers up Bézier Curves*](https://hal.science/hal-05750929) (WSCG 2026).

In the homogeneous coordinate notation of computer-aided geometric design, the curve is written with an explicit perspective division $\mathbf C(t) = \sum_{i=0}^2 w_i B_i^2(t)\,\mathbf p_i \big/ \sum_{i=0}^2 w_i B_i^2(t)$, and recovering the implicit conic requires cross-product edge normals $\mathbf n_{ij} = \tilde{\mathbf p}_i \times \tilde{\mathbf p}_j$ on augmented vectors $\tilde{\mathbf p}_i = (x_i, y_i, 1)^\top$ assembled into the $3\times 3$ matrix $\tilde{\mathbf x}^\top \big(w_0 w_2\,\mathbf n_{02}\mathbf n_{02}^\top - 2 w_1^2(\mathbf n_{01}\mathbf n_{12}^\top + \mathbf n_{12}\mathbf n_{01}^\top)\big)\tilde{\mathbf x} = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from math import comb

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_animation
from examples.geometry.bezier import render, scenarios

mv = NumpyContext(ga).multivector
# A point of the plane is a ray from the origin of space, and a line of the plane is a plane through the origin.
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Conic = ga.gatype((Plane, Point))                   # Plane+ <- Point-

## Sweeping the cone of a quadratic curve

All rays of a quadratic Bézier curve lie on one cone determined by the control triangle alone. Joining two control points gives the plane through the origin and their connecting line, which reads any ray by the pairing `plane & ray`, and balancing the chord plane `start & end` against the symmetrized tangent planes `start & middle` and `middle & end` in the ratio of the control weights yields the cone's duality map. Because the end weights enter that map only through their product `first * last`, negating both end weights leaves the cone unchanged while tracing the complementary arc on the opposite side of the control chord.

In matrix notation on homogeneous coordinates $\tilde{\mathbf x} = (x, y, w)^\top$, the symmetric matrix reads $C = w_0 w_2\,\mathbf n_{02}\mathbf n_{02}^\top - 2 w_1^2\,(\mathbf n_{01}\mathbf n_{12}^\top + \mathbf n_{12}\mathbf n_{01}^\top)$, with $\mathbf n_{ij} = \tilde{\mathbf p}_i \times \tilde{\mathbf p}_j$ the cross-product normal of the plane through the origin and control points $i$ and $j$.

In [ ]:
n_samples = scenarios.n_samples
quadratic_points = scenarios.quadratic_points
weight_sweep = scenarios.weight_sweep
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def bernstein_basis(n_degree: int, parameters: np.ndarray) -> np.ndarray:
    """The Bernstein polynomials of degree `n_degree` at each parameter form an array of shape `[..., n_degree + 1]`."""
    indices = np.arange(n_degree + 1)
    binomials = np.array([comb(n_degree, index) for index in indices])
    return binomials * parameters[..., None] ** indices * (1 - parameters[..., None]) ** (n_degree - indices)  # [..., n_degree + 1]


def bezier_curve(controls: Point, weights: np.ndarray, parameters: np.ndarray) -> Point:
    """A rational Bézier curve sums its control points weighted by their weights and Bernstein polynomials (`[...] Point-`)."""
    n_degree = controls.shape[-1] - 1
    basis = bernstein_basis(n_degree, parameters) * weights[..., None, :]                # [..., n_degree + 1]
    return (controls[..., None, :] * basis).sum(axis=-1)                                 # [...] Point-

In [ ]:
def complementary_weights(weights: np.ndarray) -> np.ndarray:
    """The complementary arc of the same conic negates both end weights, reversing the tangents at the ends."""
    ends = np.isin(np.arange(weights.shape[-1]), [0, weights.shape[-1] - 1])
    return np.where(ends, -weights, weights)


def curve_conic(controls: Point, weights: np.ndarray) -> Conic:
    """The cone swept by a quadratic curve's rays is a symmetric duality map (`[...] Plane+ <- Point-`) built from the three sides of the control triangle."""
    start, middle, end = controls[..., 0], controls[..., 1], controls[..., 2]      # [...] Point-, Point-, Point-
    first, inner, last = weights[..., 0], weights[..., 1], weights[..., 2]
    # A line of the plane is a plane through the origin, the meet of two control points.
    chord, before, after = start & end, start & middle, middle & end               # [...] Plane+, Plane+, Plane+
    # The plane through the ends balanced against the symmetrized pair of tangent planes at the ends.
    return first * last * chord * (chord & Point) - 2 * inner**2 * (before * (after & Point) + after * (before & Point))  # [...] Plane+ <- Point-


parameters = np.linspace(0.0, 1.0, n_samples + 1)                                                                       # [n_samples + 1]
# The middle weight sweeps out and back along a cosine of its logarithm so the loop turns smoothly.
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
middle_weights = np.exp(np.log(weight_sweep).mean() - np.log(weight_sweep[1] / weight_sweep[0]) / 2 * np.cos(phases))   # [n_frames]
sweep_weights = np.stack([np.ones_like(middle_weights), middle_weights, np.ones_like(middle_weights)], axis=-1)         # [n_frames, 3]
# Points (x, y) at unit weight in the plane z = 1 are the duals of x * mv.x + y * mv.y + mv.z.
controls = (mv.x * quadratic_points[:, 0] + mv.y * quadratic_points[:, 1] + mv.z).dual()                                # [3] Point-
weighted_controls = stack([controls * sweep_weights, controls * complementary_weights(sweep_weights)], axis=1)          # [n_frames, 2, 3] Point-
curves = bezier_curve(controls, sweep_weights, parameters)                                                              # [n_frames, n_samples + 1] Point-
complements = bezier_curve(controls, complementary_weights(sweep_weights), parameters)                                  # [n_frames, n_samples + 1] Point-
conics = curve_conic(controls, sweep_weights)                                                                           # [n_frames] Plane+ <- Point-

show_animation(render.animate_rays(weighted_controls, curves, complements, conics), "bezier_cone", frame_duration_ms);

Blue draws the primary Bézier curve in space together with its translucent ray fan from the origin and its normalized projection on the unit sphere, while orange draws the complementary arc obtained by negating both end weights. Grey dots mark the weighted control points along their rays from the origin, with dotted segments showing the two weighted control polygons; when the middle weight drops below one, the middle control point sits inside the sphere and its grey ray continues outward to pierce the surface. As the middle weight sweeps between 0.3 and 2, the grey loops where the cone meets the sphere stay clear of the dark equator of zero-weight points for an ellipse, touch it at unit middle weight for a parabola, and cross it for a hyperbola.

In [ ]:
# checks
# Both arcs lie on the cone of their conic at every middle weight, and the conic polarity map is symmetric.
assert ((conics[:, None](curves) & curves).abs() < 1e-10).all()
assert ((conics[:, None](complements) & complements).abs() < 1e-10).all()
assert (((conics[:, None](curves) & complements) - (conics[:, None](complements) & curves)).abs() < 1e-12).all()